# 06. Ridge com features originais — dias de atraso por contrato

**Objetivo:** testar se as colunas **originais** de `vw_modelagem_contratos_filtros`, sem nenhuma feature criada, ajudam a prever `target_dias_atraso`.

- **Alvo** (igual ao notebook 05): `max(0, duração real − prazo ajustado)`, em que a duração real é `data_recebimento_provisorio − data_ordem_servico` e o prazo ajustado soma os dias dos aditivos de prazo.
- **Baseline:** mediana do atraso no treino.
- **Modelo:** Ridge.
- **Validação:** 4 janelas temporais expanding (20→5, 25→5, 30→5, 35→5) nos 40 contratos mais antigos e teste final nos 8 mais recentes, ordenados por `data_recebimento_provisorio`.

As colunas de prazo e datas são usadas **somente para construir o alvo**, nunca como feature.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

N_TEST = 8
MIN_TRAIN = 20
BLOCK = 5
N_FOLDS = 4
RIDGE_ALPHA = 1.0

# Se os CSVs estiverem em outra pasta, defina DATA_DIR manualmente.
DATA_DIR = None

## 1. Carga dos dados

In [2]:
F_CONTRATOS = "vw_modelagem_contratos_filtros.csv"
F_ADITIVOS = "vw_taditivoprazo.csv"


def find_data_dir():
    if DATA_DIR is not None:
        return Path(DATA_DIR)
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "data", cwd / "data" / "raw", Path("/mnt/user-data/uploads")]
    for parent in cwd.parents:
        candidates += [parent / "data", parent / "data" / "raw"]
    for folder in candidates:
        if (folder / F_CONTRATOS).exists():
            return folder
    raise FileNotFoundError("CSVs não encontrados; defina DATA_DIR.")


base = find_data_dir()

contratos = pd.read_csv(
    base / F_CONTRATOS, encoding="utf-8-sig",
    parse_dates=["data_ordem_servico", "data_recebimento_provisorio"],
)
aditivos = pd.read_csv(
    base / F_ADITIVOS, encoding="utf-8-sig",
    parse_dates=["Data_termino_anterior", "Data_termino_nova"],
)

assert contratos["idContrato"].is_unique
print("Pasta de dados:", base)
print("Contratos:", contratos.shape)

Pasta de dados: /home/claude/work
Contratos: (48, 50)


## 2. Alvo com ajuste por aditivos de prazo

O fim original é `data_ordem_servico + prazo_total`. Quando há aditivo de prazo, o fim ajustado é a maior `Data_termino_nova` do contrato (término vigente). Os aditivos de outros contratos são ignorados pelo `merge`.

In [3]:
df = contratos.copy()

df["fim_original"] = df["data_ordem_servico"] + pd.to_timedelta(df["prazo_total"], unit="D")

fim_aditado = aditivos.groupby("idContrato")["Data_termino_nova"].max().rename("fim_aditado")
df = df.merge(fim_aditado, on="idContrato", how="left")

df["fim_ajustado"] = df[["fim_original", "fim_aditado"]].max(axis=1)
df["prazo_ajustado"] = (df["fim_ajustado"] - df["data_ordem_servico"]).dt.days
df["duracao_real"] = (df["data_recebimento_provisorio"] - df["data_ordem_servico"]).dt.days

df["atraso_bruto"] = df["duracao_real"] - df["prazo_ajustado"]
df["target_dias_atraso"] = df["atraso_bruto"].clip(lower=0)

print("Contratos com aditivo de prazo:", int(df["fim_aditado"].notna().sum()))
print("Contratos com atraso:", int((df["target_dias_atraso"] > 0).sum()), "de", len(df))
display(df["target_dias_atraso"].describe().to_frame().T)

Contratos com aditivo de prazo: 37
Contratos com atraso: 17 de 48


,count,mean,std,min,25%,50%,75%,max
target_dias_atraso,48.000,31.104,73.405,0.000,0.000,0.000,25.750,380.000


## 3. Seleção de colunas

Parte-se de **todas** as colunas originais e descartam-se as inadequadas. Cada coluna descartada tem o motivo registrado, e uma checagem garante que nenhuma coluna ficou sem classificação.

In [4]:
DESCARTADAS = [
    # Pedidas explicitamente
    ("prazo_total", "removida a pedido"),
    ("valor_total_contrato", "removida a pedido"),
    ("Valor_licitado", "removida a pedido"),
    ("Flag_ativo_emissao_OS", "removida a pedido"),
    # Identificadores
    *[(c, "identificador") for c in [
        "idContrato", "id_predio", "id_projeto_associado", "id_Licitacao", "id_fornecedor",
        "numero_contrato", "NumeroContrato", "processo_TRP", "processo_TRD",
        "SEI_contrato", "SEI_licitacao", "id_classificacao_contrato",
    ]],
    # Constantes nos 48 contratos
    *[(c, "constante") for c in [
        "id_diretoria", "Instrumento_contratual", "tipo_de_contrato", "id_tipo_contrato",
        "id_tipo_de_contrato", "Informacoes_para_relatorio_proj", "Possui_etapas_projeto",
        "Numero_etapas_projeto", "Pauta_GEP", "Status_contrato", "CT_paralisado",
        "CT_concluido", "CT_rescindido", "Data_paralisacao", "Motivo_paralisacao",
        "Data_previsao_retomada",
    ]],
    # Só existem depois da obra: vazamento do resultado
    *[(c, "informação pós-obra (vazamento)") for c in [
        "data_conclusao", "observacao_conclusao", "data_recebimento_provisorio",
        "data_assinatura_TRP", "data_recebimento_TRD", "data_assinatura_TRD", "observacao_contrato",
    ]],
    # Datas brutas: só entrariam como feature após criar variáveis derivadas
    *[(c, "data bruta (usada só no alvo ou descartada)") for c in [
        "data_assinatura", "data_proposta", "DataOrcamento", "data_ordem_servico", "data_ref_reajuste",
    ]],
    # Texto livre ou praticamente sem informação
    ("Descricao", "texto livre"),
    ("email_fornecedor_contrato", "texto sem informação útil"),
    # Proxy de tempo
    ("ano_contrato", "proxy de tempo (extrapola mal na validação temporal)"),
]

NUMERIC_FEATURES = ["Perc_desconto"]
CATEGORICAL_FEATURES = ["Classificacao_contrato", "id_setor_divisao"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

descartadas = pd.DataFrame(DESCARTADAS, columns=["coluna", "motivo"])
colunas_originais = set(contratos.columns)

assert descartadas["coluna"].is_unique
assert not set(descartadas["coluna"]) & set(FEATURES)
assert set(descartadas["coluna"]) | set(FEATURES) == colunas_originais, "há coluna sem classificação"

print(f"{len(colunas_originais)} colunas originais -> {len(FEATURES)} features, {len(descartadas)} descartadas\n")
print("Features usadas:", FEATURES)
display(descartadas.groupby("motivo")["coluna"].apply(lambda s: ", ".join(s)).to_frame())

50 colunas originais -> 3 features, 47 descartadas

Features usadas: ['Perc_desconto', 'Classificacao_contrato', 'id_setor_divisao']


,coluna
motivo,
constante,"id_diretoria, Instrumento_contratual, tipo_de_contrato, id_tipo_contrato, id_tipo_de_contrato, Informacoes_para_rela..."
data bruta (usada só no alvo ou descartada),"data_assinatura, data_proposta, DataOrcamento, data_ordem_servico, data_ref_reajuste"
identificador,"idContrato, id_predio, id_projeto_associado, id_Licitacao, id_fornecedor, numero_contrato, NumeroContrato, processo_..."
informação pós-obra (vazamento),"data_conclusao, observacao_conclusao, data_recebimento_provisorio, data_assinatura_TRP, data_recebimento_TRD, data_a..."
proxy de tempo (extrapola mal na validação temporal),ano_contrato
removida a pedido,"prazo_total, valor_total_contrato, Valor_licitado, Flag_ativo_emissao_OS"
texto livre,Descricao
texto sem informação útil,email_fornecedor_contrato


In [5]:
display(df[NUMERIC_FEATURES].describe().T)
for col in CATEGORICAL_FEATURES:
    print(df[col].value_counts().to_dict())

,count,mean,std,min,25%,50%,75%,max
Perc_desconto,48.000,3.458,7.133,0.000,0.000,0.000,4.067,31.660


{'Reforma Parcial': 38, 'Construção': 6, 'Reforma Parcial e Ampliação': 4}
{10: 41, 12: 5, 8: 1, 32: 1}


## 4. Separação desenvolvimento × teste final

In [6]:
df = df.sort_values(["data_recebimento_provisorio", "idContrato"]).reset_index(drop=True)

dev = df.iloc[:-N_TEST].copy()
test = df.iloc[-N_TEST:].copy()

assert dev["data_recebimento_provisorio"].max() <= test["data_recebimento_provisorio"].min()

folds = []
for k in range(N_FOLDS):
    n_train = MIN_TRAIN + k * BLOCK
    folds.append({"fold": k + 1, "train": dev.iloc[:n_train], "val": dev.iloc[n_train:n_train + BLOCK]})

print("Contratos de teste:", test["idContrato"].tolist())
display(pd.DataFrame([
    {
        "fold": f["fold"],
        "treino": len(f["train"]),
        "validacao": len(f["val"]),
        "mediana_treino": f["train"]["target_dias_atraso"].median(),
        "atraso_medio_validacao": f["val"]["target_dias_atraso"].mean(),
    }
    for f in folds
]))

Contratos de teste: [173, 201, 287, 283, 342, 317, 254, 377]


,fold,treino,validacao,mediana_treino,atraso_medio_validacao
0,1,20,5,0.000,30.000
1,2,25,5,0.000,63.800
2,3,30,5,0.000,0.000
3,4,35,5,0.000,17.800


## 5. Modelos e métricas

- **Baseline:** `DummyRegressor` com a mediana do treino.
- **Ridge:** `Perc_desconto` padronizado; `Classificacao_contrato` e `id_setor_divisao` em one-hot. Previsões negativas viram zero.

In [7]:
def make_baseline():
    return DummyRegressor(strategy="median")


def make_ridge():
    prep = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("prep", prep), ("model", Ridge(alpha=RIDGE_ALPHA))])


def predict(model, frame):
    return np.maximum(0, model.predict(frame[FEATURES]))


def metrics(y_true, y_pred, full=False):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    atrasados = y_true > 0

    out = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae_so_atrasados": mean_absolute_error(y_true[atrasados], y_pred[atrasados]) if atrasados.any() else np.nan,
        "vies_medio": (y_pred - y_true).mean(),
    }
    if full:
        out["r2"] = r2_score(y_true, y_pred)
        out["spearman"] = spearmanr(y_true, y_pred)[0] if np.unique(y_pred).size > 1 else np.nan
    return out

## 6. Validação temporal (apenas desenvolvimento)

In [8]:
cv_rows, oof_rows = [], []

for f in folds:
    tr, va = f["train"], f["val"]
    baseline = make_baseline().fit(tr[FEATURES], tr["target_dias_atraso"])
    ridge = make_ridge().fit(tr[FEATURES], tr["target_dias_atraso"])

    preds = {"baseline_mediana": predict(baseline, va), "ridge": predict(ridge, va)}

    for name, pred in preds.items():
        cv_rows.append({"fold": f["fold"], "modelo": name, **metrics(va["target_dias_atraso"], pred)})

    oof_rows.append(pd.DataFrame({
        "fold": f["fold"], "id_contrato": va["idContrato"].values,
        "real": va["target_dias_atraso"].values, **preds,
    }))

cv = pd.DataFrame(cv_rows)
oof = pd.concat(oof_rows, ignore_index=True)

print("MAE por janela:")
display(cv.pivot(index="fold", columns="modelo", values="mae"))

print("Resumo entre janelas:")
cv_resumo = cv.groupby("modelo").agg(mae_medio=("mae", "mean"), mae_dp=("mae", "std"), rmse_medio=("rmse", "mean"))
display(cv_resumo)

print("Métricas agregadas (20 contratos de validação):")
display(pd.DataFrame({n: metrics(oof["real"], oof[n], full=True) for n in ["baseline_mediana", "ridge"]}).T)

MAE por janela:


modelo,baseline_mediana,ridge
fold,,
1,30.000,32.306
2,63.800,70.930
3,0.000,51.008
4,17.800,108.433


Resumo entre janelas:


,mae_medio,mae_dp,rmse_medio
modelo,,,
baseline_mediana,27.900,26.917,52.053
ridge,65.669,32.580,84.869


Métricas agregadas (20 contratos de validação):


,mae,rmse,mae_so_atrasados,vies_medio,r2,spearman
baseline_mediana,27.900,69.787,79.714,-27.900,-0.190,NaN
ridge,65.669,94.087,77.108,30.342,-1.163,-0.197


## 7. Teste final

In [9]:
baseline_final = make_baseline().fit(dev[FEATURES], dev["target_dias_atraso"])
ridge_final = make_ridge().fit(dev[FEATURES], dev["target_dias_atraso"])

test_pred = pd.DataFrame({
    "id_contrato": test["idContrato"].values,
    "real": test["target_dias_atraso"].values,
    "baseline_mediana": predict(baseline_final, test),
    "ridge": predict(ridge_final, test),
})

print("Mediana do baseline:", baseline_final.constant_.ravel()[0])
display(pd.DataFrame({n: metrics(test_pred["real"], test_pred[n], full=True) for n in ["baseline_mediana", "ridge"]}).T)
display(test_pred)

mae_b = mean_absolute_error(test_pred["real"], test_pred["baseline_mediana"])
mae_r = mean_absolute_error(test_pred["real"], test_pred["ridge"])
print(f"CV   — MAE médio: baseline {cv_resumo.loc['baseline_mediana', 'mae_medio']:.1f} | Ridge {cv_resumo.loc['ridge', 'mae_medio']:.1f}")
print(f"Teste — MAE:      baseline {mae_b:.1f} | Ridge {mae_r:.1f}")

Mediana do baseline: 0.0


,mae,rmse,mae_so_atrasados,vies_medio,r2,spearman
baseline_mediana,21.875,54.352,87.500,-21.875,-0.193,NaN
ridge,40.849,58.080,52.319,16.914,-0.363,0.298


,id_contrato,real,baseline_mediana,ridge
0,173,0,0.000,109.737
1,201,0,0.000,0.000
2,287,0,0.000,50.948
3,283,0,0.000,55.377
4,342,0,0.000,6.089
5,317,23,0.000,31.901
6,254,0,0.000,0.000
7,377,152,0.000,56.263


CV   — MAE médio: baseline 27.9 | Ridge 65.7
Teste — MAE:      baseline 21.9 | Ridge 40.8


## 8. Coeficientes do Ridge (modelo final)

In [10]:
coef = pd.Series(
    ridge_final["model"].coef_,
    index=ridge_final["prep"].get_feature_names_out(),
).sort_values(key=np.abs, ascending=False)

print("Intercepto:", round(ridge_final["model"].intercept_, 2))
display(coef.to_frame("coeficiente"))

Intercepto: 0.81


,coeficiente
cat__id_setor_divisao_12,-47.288
cat__id_setor_divisao_10,31.809
num__Perc_desconto,29.041
cat__Classificacao_contrato_Construção,-24.269
cat__id_setor_divisao_8,19.954
cat__Classificacao_contrato_Reforma Parcial e Ampliação,13.602
cat__Classificacao_contrato_Reforma Parcial,10.668
cat__id_setor_divisao_32,-4.475


## Leitura dos resultados

**O Ridge com as features originais não supera o baseline.**

| | MAE validação temporal (média de 4 janelas) | MAE teste final (8 contratos) |
|---|---|---|
| Baseline (mediana = 0 dia) | 27,9 | 21,9 |
| Ridge | 65,7 | 40,8 |

- O Ridge fica pior que o baseline em **todas as 4 janelas** (a pior é a 4, com MAE de 108 contra 18 do baseline).
- Sobraram só **3 colunas utilizáveis** (`Perc_desconto`, `Classificacao_contrato`, `id_setor_divisao`). Sem prazo, valor e datas, o modelo tem pouca informação sobre a obra.
- `id_setor_divisao` é um identificador com categorias de 1 contrato (setores 8 e 32). O one-hot dessas categorias só memoriza esses contratos, e os maiores coeficientes do modelo são justamente de setor (12 e 10).
- `Perc_desconto` é 0 em todos os contratos anteriores a out/2020 e só passa a variar depois. Ela pode estar capturando a época, e não uma característica da obra.
- No teste, o Spearman do Ridge é positivo (0,30), mas com 8 contratos e só 2 atrasados isso não é conclusivo, e o erro absoluto continua maior que o do baseline.
- O baseline prevê 0 sempre. Ele vence no MAE porque 31 dos 48 contratos não atrasam, mas não ajuda a priorizar contratos.

**Ideias para testar depois (não executadas aqui):** retirar `id_setor_divisao`, aumentar `RIDGE_ALPHA` (ou usar `RidgeCV`), ou reincluir `prazo_total`, que é a coluna original mais ligada ao tamanho do atraso em dias.